# OpsPilot — Phase 2: Baseline Rules-Based Agent
**Capstone | Scenario 1: IT Operations Copilot | Track A: LangChain**

This notebook builds a rules-based (no-LLM) baseline agent that:
- Accepts natural language questions about IT incidents and services
- Uses keyword matching and Pandas to query synthetic data
- Demonstrates clear safety (refusing action requests)
- Exposes 5 concrete limitations that motivate Phase 3 (LLM integration)

---

## 1. Install Dependencies

In [ ]:
# Run this cell once — installs all packages needed for Phase 2
%pip install pandas --quiet
print('✅ Dependencies ready')

## 2. Generate Synthetic Data
Creates `incidents.csv`, `services.csv`, `sla_targets.csv`, and runbook text files.

In [ ]:
import csv, random, os
from datetime import datetime, timedelta

random.seed(42)
os.makedirs('data/runbooks', exist_ok=True)
os.makedirs('logs', exist_ok=True)

SERVICES = [
    'auth-service', 'payments-api', 'user-portal', 'notification-service',
    'database-cluster', 'api-gateway', 'reporting-engine',
    'inventory-service', 'search-service', 'email-relay',
]
SEVERITIES    = ['P1', 'P2', 'P3', 'P4']
SEV_WEIGHTS   = [0.08, 0.22, 0.45, 0.25]
ROOT_CAUSES   = [
    'Memory leak', 'Database connection exhaustion', 'Certificate expiry',
    'Upstream dependency failure', 'Misconfiguration after deploy',
    'DDoS attack pattern', 'Disk space exhaustion', 'Network packet loss',
    'CPU spike from batch job', 'Third-party API timeout',
]
STATUSES      = ['Resolved', 'Resolved', 'Resolved', 'Open', 'In Progress']
ANALYSTS      = ['ANL-001', 'ANL-002', 'ANL-003', 'ANL-004', 'ANL-005']
SLA_MTTR      = {'P1': 60, 'P2': 240, 'P3': 1440, 'P4': 4320}
BASE_DATE     = datetime(2025, 1, 1)
END_DATE      = datetime(2026, 5, 28)

def random_dt(s, e):
    d = e - s
    return s + timedelta(seconds=random.randint(0, int(d.total_seconds())))

def mttr(sev, sla):
    if random.random() < 0.30:
        return sla + random.randint(10, sla)
    return random.randint(max(5, sla // 4), sla - 5)

# incidents.csv
rows = []
for i in range(1, 501):
    sev   = random.choices(SEVERITIES, weights=SEV_WEIGHTS)[0]
    sla   = SLA_MTTR[sev]
    m     = mttr(sev, sla)
    svc   = random.choice(SERVICES)
    cause = random.choice(ROOT_CAUSES)
    opn   = random_dt(BASE_DATE, END_DATE)
    stat  = random.choice(STATUSES)
    res   = opn + timedelta(minutes=m) if stat == 'Resolved' else ''
    mm    = m if stat == 'Resolved' else ''
    if i % 7  == 0: svc = 'auth-service'
    if i % 11 == 0: svc = 'payments-api'
    rows.append({
        'incident_id':  f'INC-{i:04d}', 'service': svc, 'severity': sev,
        'status': stat, 'root_cause': cause,
        'opened_at':   opn.strftime('%Y-%m-%d %H:%M:%S'),
        'resolved_at': res.strftime('%Y-%m-%d %H:%M:%S') if res else '',
        'mttr_minutes': mm, 'sla_minutes': sla,
        'sla_breached': 'Yes' if mm and int(mm) > sla else 'No',
        'assigned_to':  random.choice(ANALYSTS),
        'notes':        f'Incident on {svc}. Root cause: {cause}.',
    })

with open('data/incidents.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print(f'✅ incidents.csv — {len(rows)} rows')

# services.csv
svc_rows = []
for svc in SERVICES:
    svc_rows.append({
        'service_name': svc,
        'team_owner': random.choice(['Platform','Payments','Identity','Data','Core']),
        'environment': 'Production',
        'uptime_pct_30d': round(random.uniform(95.5, 99.99), 2),
        'avg_mttr_minutes': random.randint(25, 180),
        'open_incidents': random.randint(0, 4),
        'last_p1_date': random_dt(datetime(2025,10,1), END_DATE).strftime('%Y-%m-%d'),
        'criticality': random.choice(['Critical','High','Medium']),
    })
with open('data/services.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=svc_rows[0].keys())
    w.writeheader(); w.writerows(svc_rows)
print(f'✅ services.csv — {len(svc_rows)} rows')

# sla_targets.csv
sla_rows = [
    {'severity':'P1','sla_mttr_minutes':60, 'description':'Critical — full outage'},
    {'severity':'P2','sla_mttr_minutes':240,'description':'High — major degradation'},
    {'severity':'P3','sla_mttr_minutes':1440,'description':'Medium — partial impact'},
    {'severity':'P4','sla_mttr_minutes':4320,'description':'Low — cosmetic / minor'},
]
with open('data/sla_targets.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=sla_rows[0].keys())
    w.writeheader(); w.writerows(sla_rows)
print(f'✅ sla_targets.csv — {len(sla_rows)} rows')

# Runbook text files (used in Phase 4 RAG)
runbooks = {
  'data/runbooks/auth-service.txt': '''AUTH-SERVICE RUNBOOK — NovaTech IT Operations
Service: auth-service | Owner: Identity Team | Criticality: Critical
COMMON FAILURES:
1. Certificate Expiry — rotate via PKI portal (approval required). SLA: P1 60 min.
2. Memory Leak — rolling restart needs ops-lead approval. Often post-deploy.
3. DB Connection Exhaustion — kill stale connections; escalate to DB team in 20 min.
ESCALATION: L1→L2 (15min)→Ops Lead (30min)→Identity On-Call (45min)
KNOWN ISSUES: Cert renewal missed Jan 2025 (47-min P1). Memory leak in v3.2.1 (patched v3.2.3).''',
  'data/runbooks/payments-api.txt': '''PAYMENTS-API RUNBOOK — NovaTech IT Operations
Service: payments-api | Owner: Payments Team | Criticality: Critical
COMMON FAILURES:
1. Third-Party API Timeout — enable fallback processor (payments team required). SLA: P1 60 min.
2. DB Connection Exhaustion — scale pool or restart pool manager.
3. DDoS — enable WAF rate limiting, page security team.
ESCALATION: L1→L2 (10min)→Payments On-Call (20min)→Payments Eng Lead (30min)
IMPORTANT: P1 outages must be communicated to Finance within 15 minutes.''',
  'data/runbooks/general_ops.txt': '''GENERAL OPS RUNBOOK — NovaTech IT Operations
SLA: P1=60min, P2=4hr, P3=24hr, P4=72hr (MTTR targets)
ESCALATION: If MTTR will exceed SLA, escalate immediately. P1 needs Ops Lead in 10min.
SHIFT HANDOFF: List open incidents, SLA-at-risk items, patterns observed.
DATA POLICY: NOC analysts are READ-ONLY. All remediation needs Ops Lead approval.
Actions (restart/config/rollback) performed by on-call engineer only.''',
}
for path, content in runbooks.items():
    with open(path, 'w') as f: f.write(content)
    print(f'✅ {path}')

print('\n🎉 All synthetic data ready!')

## 3. Build the Baseline Agent

In [ ]:
import re, json, logging
import pandas as pd
from datetime import datetime, timedelta

# ── PII-Safe Logging ─────────────────────────────────────────────────────────
logging.basicConfig(
    filename='logs/baseline_interactions.log', level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)

def _strip_pii(text):
    text = re.sub(r'\bANL-\d{3}\b', '[ANALYST]', text)
    text = re.sub(r'\b[A-Z][a-z]+ [A-Z][a-z]+\b', '[NAME]', text)
    return text

def log_interaction(query, response, ms):
    logging.info(json.dumps({
        'query': _strip_pii(query), 'response': _strip_pii(response[:200]),
        'duration_ms': round(ms, 1), 'agent': 'baseline-v1'
    }))

# ── Data Loading ──────────────────────────────────────────────────────────────
incidents = pd.read_csv('data/incidents.csv')
incidents['opened_at']    = pd.to_datetime(incidents['opened_at'],    errors='coerce')
incidents['resolved_at']  = pd.to_datetime(incidents['resolved_at'],  errors='coerce')
incidents['mttr_minutes'] = pd.to_numeric(incidents['mttr_minutes'],  errors='coerce')
services    = pd.read_csv('data/services.csv')
sla_targets = pd.read_csv('data/sla_targets.csv')
data = {'incidents': incidents, 'services': services, 'sla': sla_targets}
print(f'✅ {len(incidents)} incidents | {len(services)} services | {len(sla_targets)} SLA rules loaded')

In [ ]:
# ── Keyword Rules ──────────────────────────────────────────────────────────────
RULES = [
    (r'\b(how many|count)\b.*(incident|p1|p2|p3|p4)', 'count_incidents'),
    (r'\bsla\b|\bbreach(es|ed)?\b',                   'sla_breaches'),
    (r'\b(top|root|common)\b.*(cause|causes)\b',      'top_root_causes'),
    (r'\b(mttr|resolution time|mean time)\b',          'avg_mttr'),
    (r'\b(open|ongoing|active)\b.*(incident)',         'open_incidents'),
    (r'\b(uptime|availability)\b',                     'service_uptime'),
    (r'\b(restart|reboot|shutdown|kill|deploy|modif|change|update)\b', 'refuse_action'),
    (r'\b(help|what can you|commands)\b',              'show_help'),
    (r'\binc-\d{4}\b',                                'lookup_incident'),
]

def match_rule(query):
    q = query.lower()
    for pattern, handler in RULES:
        if re.search(pattern, q):
            return handler
    return 'no_match'

def _extract_service(query):
    for svc in services['service_name'].tolist():
        if svc.lower() in query.lower():
            return svc
    return None

# ── Handlers ──────────────────────────────────────────────────────────────────
def count_incidents(query):
    df = incidents.copy()
    sev = next((s for s in ['P1','P2','P3','P4'] if s.lower() in query.lower()), None)
    m = re.search(r'last (\d+) day', query.lower())
    if m:
        df = df[df['opened_at'] >= datetime.now() - timedelta(days=int(m.group(1)))]
        period = f'last {m.group(1)} days'
    else:
        period = 'all time'
    if sev:
        df = df[df['severity'] == sev]
        return (f'📊 {sev} incidents ({period}): {len(df)}\n'
                f'   Resolved: {(df["status"]=="Resolved").sum()}  |  Open: {(df["status"]!="Resolved").sum()}')
    counts = df.groupby('severity').size().reindex(['P1','P2','P3','P4'], fill_value=0)
    return '\n'.join([f'📊 Incident count ({period}):'] +
                     [f'   {k}: {v}' for k, v in counts.items()] +
                     [f'   Total: {counts.sum()}'])

def sla_breaches(query):
    df = incidents[incidents['sla_breached'] == 'Yes']
    sev = next((s for s in ['P1','P2','P3','P4'] if s.lower() in query.lower()), None)
    if sev: df = df[df['severity'] == sev]
    by_svc = df.groupby('service').size().sort_values(ascending=False).head(5)
    lines = [f"⚠️  SLA Breaches{' (' + sev + ')' if sev else ''}: {len(df)} total",
             '   Top services by breach count:']
    lines += [f'   • {s}: {c}' for s, c in by_svc.items()]
    return '\n'.join(lines)

def top_root_causes(query):
    df = incidents.copy()
    sev = next((s for s in ['P1','P2','P3','P4'] if s.lower() in query.lower()), None)
    if sev: df = df[df['severity'] == sev]
    top = df['root_cause'].value_counts().head(5)
    lines = [f"🔍 Top root causes{' for ' + sev if sev else ''}:"]
    lines += [f'   {i+1}. {c} — {n} incidents' for i, (c, n) in enumerate(top.items())]
    return '\n'.join(lines)

def avg_mttr(query):
    df = incidents[incidents['mttr_minutes'].notna()].copy()
    svc = _extract_service(query)
    if svc: df = df[df['service'] == svc]
    label = svc or 'all services'
    by_sev = df.groupby('severity')['mttr_minutes'].mean().reindex(['P1','P2','P3','P4'])
    sla_idx = sla_targets.set_index('severity')
    lines = [f'⏱️  Average MTTR for {label}:']
    for s, v in by_sev.items():
        if pd.isna(v):
            lines.append(f'   {s}: no data')
        else:
            t = sla_idx.loc[s, 'sla_mttr_minutes']
            lines.append(f'   {s}: {v:.0f} min (target {t} min) {"⛔ over SLA" if v > t else "✅ within SLA"}')
    return '\n'.join(lines)

def open_incidents(query):
    df = incidents[incidents['status'].isin(['Open','In Progress'])].copy()
    svc = _extract_service(query)
    label = svc or 'all services'
    if svc: df = df[df['service'] == svc]
    if df.empty: return f'✅ No open incidents for {label}.'
    lines = [f'🔴 Open incidents for {label}: {len(df)}']
    for _, r in df.head(5).iterrows():
        lines.append(f"   • {r['incident_id']} | {r['severity']} | {r['service']} | opened {r['opened_at'].strftime('%Y-%m-%d %H:%M')}")
    if len(df) > 5: lines.append(f'   ... and {len(df)-5} more.')
    return '\n'.join(lines)

def service_uptime(query):
    svc = _extract_service(query)
    if svc:
        row = services[services['service_name'] == svc]
        if row.empty: return f"⚠️  No uptime data for '{svc}'."
        r = row.iloc[0]
        return (f'📡 {svc} uptime (30d): {r["uptime_pct_30d"]}%\n'
                f'   Avg MTTR: {r["avg_mttr_minutes"]} min | Open: {r["open_incidents"]} | Criticality: {r["criticality"]}')
    lines = ['📡 Service uptime summary (30d):']
    for _, r in services.sort_values('uptime_pct_30d').iterrows():
        flag = '⚠️ ' if r['uptime_pct_30d'] < 99.0 else '  '
        lines.append(f"   {flag}{r['service_name']}: {r['uptime_pct_30d']}%")
    return '\n'.join(lines)

def lookup_incident(query):
    m = re.search(r'inc-(\d{4})', query.lower())
    if not m: return '❓ Could not extract incident ID. Format: INC-XXXX'
    inc_id = f'INC-{m.group(1)}'
    row = incidents[incidents['incident_id'] == inc_id]
    if row.empty: return f'⚠️  Incident {inc_id} not found.'
    r = row.iloc[0]
    status_line = (f'Resolved at {r["resolved_at"].strftime("%Y-%m-%d %H:%M")} (MTTR: {int(r["mttr_minutes"])} min)'
                   if r['status'] == 'Resolved' else f'Status: {r["status"]}')
    return '\n'.join([f'📋 {inc_id}', f'   Service  : {r["service"]}',
                      f'   Severity : {r["severity"]}', f'   Root cause: {r["root_cause"]}',
                      f'   Opened   : {r["opened_at"].strftime("%Y-%m-%d %H:%M")}',
                      f'   {status_line}', f'   SLA breach: {r["sla_breached"]}'])

def refuse_action(query):
    return ('🚫 I am a read-only Decision Support copilot.\n'
            '   I cannot restart services, modify configuration, trigger deployments,\n'
            '   or take any operational action.\n'
            '   Please escalate to your on-call engineer.\n'
            '   I can help analyse incident history or SLA data to support your decision.')

def show_help(query):
    return ('🤖 OpsPilot Baseline — Supported queries:\n'
            '   • How many incidents / P1s in the last N days?\n'
            '   • Which services had SLA breaches?\n'
            '   • What are the top root causes for P1 incidents?\n'
            '   • What is the MTTR for [service]?\n'
            '   • Are there open incidents for [service]?\n'
            '   • What is the uptime for [service]?\n'
            '   • Tell me about INC-XXXX')

def no_match(query):
    return ('❓ I did not understand that query.\n'
            '   This is a known limitation — keyword rules cannot handle\n'
            '   paraphrases, causal questions, or complex multi-part queries.\n'
            '   Type \'help\' to see supported question types.')

HANDLERS = {
    'count_incidents': count_incidents, 'sla_breaches': sla_breaches,
    'top_root_causes': top_root_causes, 'avg_mttr': avg_mttr,
    'open_incidents':  open_incidents,  'service_uptime': service_uptime,
    'lookup_incident': lookup_incident, 'refuse_action': refuse_action,
    'show_help': show_help, 'no_match': no_match,
}

def respond(query):
    t0 = datetime.now()
    fn = HANDLERS[match_rule(query)]
    result = fn(query)
    ms = (datetime.now() - t0).total_seconds() * 1000
    log_interaction(query, result, ms)
    return result

print('✅ Baseline agent loaded — all handlers registered')

## 4. Demo Runs — Working Queries

In [ ]:
WORKING_QUERIES = [
    'How many P1 incidents in the last 30 days?',
    'Which services had the most SLA breaches?',
    'What are the top root causes for P1 incidents?',
    'What is the average MTTR for auth-service?',
    'Are there any open incidents for payments-api?',
    'Tell me about INC-0042',
    'What is the uptime for database-cluster?',
]

print('=' * 60)
print('WORKING QUERIES — Baseline Agent Handles These Correctly')
print('=' * 60)
for i, q in enumerate(WORKING_QUERIES, 1):
    print(f'\n[Q{i:02d}] {q}')
    print('-' * 50)
    print(respond(q))

## 5. Safety Demo — Refuses Action Requests

In [ ]:
SAFETY_QUERIES = [
    'Restart the auth-service immediately',
    'Deploy the hotfix to payments-api',
]

print('=' * 60)
print('SAFETY TESTS — Agent MUST Refuse These Requests')
print('=' * 60)
for i, q in enumerate(SAFETY_QUERIES, 1):
    print(f'\n[S{i}] {q}')
    print('-' * 50)
    print(respond(q))

## 6. Limitations Demo — Why Baseline Is Insufficient

In [ ]:
LIMITATION_QUERIES = [
    ('L1 — Vague time reference',     'Is the system behaving unusually lately?'),
    ('L2 — Multi-part question',      'Give me a full health report for this week'),
    ('L3 — Causal reasoning',         'Why did payments-api spike last Tuesday?'),
]

print('=' * 60)
print('LIMITATIONS — Queries the Baseline Agent CANNOT Handle')
print('=' * 60)
for label, q in LIMITATION_QUERIES:
    print(f'\n[{label}]')
    print(f'Query: {q}')
    print('-' * 50)
    print(respond(q))

print('\n' + '=' * 60)
print('SUMMARY OF LIMITATIONS:')
print('  L1 — Cannot handle vague/implicit time ranges')
print('  L2 — Cannot compose multi-part answers')
print('  L3 — No causal reasoning ability')
print('  L4 — No memory: each query is fully stateless')
print('  L5 — Cannot express uncertainty — just returns no_match')
print('  → These gaps motivate Phase 3: LLM Integration')
print('=' * 60)

## 7. Verify PII-Safe Logs

In [ ]:
print('Last 5 log entries (PII-safe):')
print('-' * 60)
with open('logs/baseline_interactions.log') as f:
    lines = f.readlines()
for line in lines[-5:]:
    print(line.strip())
    
# Verify no analyst IDs leak
log_text = ''.join(lines)
import re
pii_found = re.findall(r'ANL-\d{3}', log_text)
print(f'\nPII check — raw analyst IDs in log: {len(pii_found)} (should be 0 ✅)')

## 8. Phase 2 Summary

| Capability | Status |
|-----------|--------|
| Accepts natural language input | ✅ |
| Queries structured CSV data | ✅ |
| Refuses action requests (safety) | ✅ |
| PII-safe logging | ✅ |
| Handles vague/paraphrased queries | ❌ L1 |
| Multi-part question handling | ❌ L2 |
| Causal / why reasoning | ❌ L3 |
| Conversational memory | ❌ L4 |
| Uncertainty expression | ❌ L5 |

**Next:** Phase 3 — Integrate an LLM to overcome L1–L5.